# 04 — Custom Metrics

This notebook shows how to:

1. Create a custom metric by subclassing `BaseMetric`
2. Register it with `register_metric()` so it can be used by name
3. Use `build_metrics()` to mix built-in and custom metrics
4. Run the custom metric through the evaluator

No API key needed — all calls use a `ReplayJudge`.

In [1]:
from pydantic import BaseModel, Field

from evalf import EvalCase, Evaluator
from evalf.llms.base import BaseLLMModel
from evalf.metrics import BaseMetric, build_metrics, list_metric_names, register_metric
from evalf.schemas import LLMResponse, UsageStats

## 1. Define the output schema

Every metric needs a Pydantic model that the judge returns.
It must have at least a `score` field (0.0–1.0).
Add a `reason` field if you want explanations.

In [2]:
class ConcisenessAssessment(BaseModel):
    """Judge output for answer conciseness."""

    score: float = Field(ge=0.0, le=1.0)
    reason: str

## 2. Subclass `BaseMetric`

You need to set:
- `name` — identifier used in reports
- `required_inputs` — fields that must be present on the `EvalCase`
- `output_schema` — the Pydantic model from step 1
- `build_prompt()` — returns `(system_prompt, user_prompt)` for the judge

In [3]:
class ConcisenessMetric(BaseMetric):
    """Score whether an answer is concise and free of unnecessary filler."""

    name = "conciseness"
    required_inputs = ("question", "actual_output")
    output_schema = ConcisenessAssessment

    def build_prompt(self, case: EvalCase) -> tuple[str, str]:
        system_prompt = (
            "You are a strict judge for answer conciseness. "
            "Score how concise and to-the-point the answer is."
        )
        user_prompt = (
            f"Question: {case.question}\n\n"
            f"Answer: {case.actual_output}\n\n"
            "Score the answer for conciseness on a scale from 0.0 to 1.0.\n"
            "1.0 = perfectly concise, no filler.\n"
            "0.0 = extremely verbose with unnecessary content.\n\n"
            'Return JSON: {"score": ..., "reason": "..."}'
        )
        return system_prompt, user_prompt

## 3. Register the custom metric

`register_metric()` makes it available to `build_metrics()`
and the CLI `--metrics` flag.

In [4]:
register_metric("conciseness", ConcisenessMetric)

print("Registered metrics:")
print(list_metric_names())

Registered metrics:
['answer_correctness', 'answer_relevance', 'c4', 'conciseness', 'context_coverage', 'context_precision', 'context_recall', 'context_relevance', 'faithfulness']


## 4. Use `build_metrics()` to mix built-in and custom

You can mix registered metric names freely.
Each metric gets its own threshold override if needed.

In [5]:
metrics = build_metrics(
    ["answer_relevance", "conciseness"],
    default_threshold=0.7,
    threshold_overrides={"conciseness": 0.6},
)

for m in metrics:
    print(f"  {m.name}: threshold={m.threshold}")

  answer_relevance: threshold=0.7
  conciseness: threshold=0.6


## 5. Run through the evaluator

In [6]:
class ReplayJudge(BaseLLMModel):
    def __init__(self, outputs):
        super().__init__(provider="test", model="replay", base_url="https://example.com", api_key=None)
        self._outputs = list(outputs)

    def _pop(self):
        return LLMResponse(
            text=None, model=self.model, provider=self.provider,
            parsed_output=self._outputs.pop(0),
            usage=UsageStats(input_tokens=15, output_tokens=8, total_tokens=23, latency_ms=10.0),
        )

    def generate(self, **kw): return self._pop()
    async def a_generate(self, **kw): return self._pop()

In [7]:
from evalf.metrics.answer_relevance.schema import AnswerRelevanceAssessment

case = EvalCase(
    id="custom-1",
    question="What does FERPA protect?",
    actual_output="FERPA protects the privacy of student education records.",
)

judge_outputs = [
    AnswerRelevanceAssessment(score=1.0, reason="Directly answers the question."),
    ConcisenessAssessment(score=0.95, reason="One clear sentence, no filler."),
]

report = await Evaluator(judge=ReplayJudge(judge_outputs), concurrency=1).a_evaluate(
    cases=[case],
    metrics=metrics,
)

for m in report.samples[0].metrics:
    print(f"  {m.name:25s}  score={m.score}  status={m.status}")

  answer_relevance           score=1.0  status=passed
  conciseness                score=0.95  status=passed


## Summary

To create a custom metric:

1. Define a Pydantic `output_schema` with `score` and optional `reason`
2. Subclass `BaseMetric`, set `name`, `required_inputs`, `output_schema`
3. Implement `build_prompt(case) -> (system_prompt, user_prompt)`
4. Call `register_metric("name", MyMetric)` to make it available by name
5. Use `build_metrics(["name"])` or pass the class directly to `Evaluator`

For multi-step metrics that need several judge calls,
subclass `BaseDecomposedMetric` and implement `compute_assessment()` /
`a_compute_assessment()` instead.